# Reading Assistant — GPU Training Experiment

This is a **research baseline**, not production-quality TTS. It trains a compact phoneme-to-fixed-length-mel predictor on a small LJSpeech subset and saves model weights, vocabulary, training metrics, and an example WAV. Run with **Runtime → Change runtime type → T4 GPU** in Colab. The original notebook is unchanged.

**Limitations:** fixed-duration output, approximate spectrogram inversion, no alignment/duration predictor. Do not interpret a successful run as natural audiobook-quality speech.

In [ ]:
%pip -q install "torchaudio" "g2p_en" "nltk" "soundfile"
import os, json, random, pathlib, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader, random_split
from g2p_en import G2p
import nltk
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("cmudict", quiet=True)
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:",device, "PyTorch:",torch.__version__)
assert device.type=="cuda", "Select a GPU runtime in Colab before running training."

In [ ]:
# Keep the first experiment small enough for a free Colab GPU.
SAMPLES=320
EPOCHS=3
BATCH_SIZE=8
MAX_PHONES=180
MEL_FRAMES=128
N_MELS=80
SAMPLE_RATE=22050
root="/content/reading_assistant_data"
raw=torchaudio.datasets.LJSPEECH(root=root, download=True)
g2p=G2p()
# Stable vocabulary; include symbols actually produced by g2p.
phones=sorted(set(g2p.phonemes)|{" "})
vocab={"<pad>":0,"<unk>":1}
for p in phones:
    if p not in vocab:vocab[p]=len(vocab)
mel_fn=torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE,n_fft=1024,hop_length=256,n_mels=N_MELS)
class SpeechDataset(Dataset):
    def __len__(self):return min(SAMPLES,len(raw))
    def __getitem__(self,idx):
        wav,sr,text,_=raw[idx]
        if sr!=SAMPLE_RATE: wav=torchaudio.functional.resample(wav,sr,SAMPLE_RATE)
        ids=[vocab.get(p,1) for p in g2p(text.lower())][:MAX_PHONES]
        ids=ids or [1]
        mel=torch.log1p(mel_fn(wav).squeeze(0))
        # Fixed-frame baseline; temporal resizing is intentionally lossy.
        mel=F.interpolate(mel[None,None],size=(N_MELS,MEL_FRAMES),mode="bilinear",align_corners=False)[0,0].T
        return torch.tensor(ids,dtype=torch.long),mel

def collate(batch):
    x=nn.utils.rnn.pad_sequence([b[0] for b in batch],batch_first=True,padding_value=0)
    y=torch.stack([b[1] for b in batch]);return x,y
full=SpeechDataset();n_val=max(1,len(full)//10)
train,val=random_split(full,[len(full)-n_val,n_val],generator=torch.Generator().manual_seed(SEED))
train_loader=DataLoader(train,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,num_workers=0)
val_loader=DataLoader(val,batch_size=BATCH_SIZE,collate_fn=collate,num_workers=0)
print("Training samples:",len(train),"Validation samples:",len(val))

In [ ]:
class BaselineTTS(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.embed=nn.Embedding(vocab_size,128,padding_idx=0)
        self.encoder=nn.GRU(128,192,batch_first=True,bidirectional=True)
        self.head=nn.Sequential(nn.Linear(384,512),nn.ReLU(),nn.Linear(512,MEL_FRAMES*N_MELS))
    def forward(self,ids):
        seq,_=self.encoder(self.embed(ids))
        mask=(ids!=0).unsqueeze(-1)
        pooled=(seq*mask).sum(1)/mask.sum(1).clamp_min(1)
        return self.head(pooled).reshape(-1,MEL_FRAMES,N_MELS)
model=BaselineTTS(len(vocab)).to(device)
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4)
metrics=[]
for epoch in range(EPOCHS):
    losses=[];model.train()
    for ids,mel in train_loader:
        ids,mel=ids.to(device),mel.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss=F.mse_loss(model(ids),mel)
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),1.0);optimizer.step()
        losses.append(loss.item())
    model.eval();validation=[]
    with torch.no_grad():
        for ids,mel in val_loader:
            validation.append(F.mse_loss(model(ids.to(device)),mel.to(device)).item())
    record={"epoch":epoch+1,"train_mse":float(np.mean(losses)),"validation_mse":float(np.mean(validation))}
    metrics.append(record);print(record)

In [ ]:
# Export artifacts to Colab and optionally copy them to Google Drive.
out=pathlib.Path("/content/reading_assistant_outputs");out.mkdir(exist_ok=True)
torch.save({"model_state_dict":model.cpu().state_dict(),"vocab_size":len(vocab),"mel_frames":MEL_FRAMES,"n_mels":N_MELS,"sample_rate":SAMPLE_RATE,"architecture":"BaselineTTS_v1"},out/"reading_assistant_model.pt")
(out/"vocabulary.json").write_text(json.dumps(vocab,indent=2))
(out/"training_metrics.json").write_text(json.dumps({"metrics":metrics,"samples":len(full),"note":"Research baseline; not a speech naturalness evaluation"},indent=2))
# Reconstruct a sample waveform from the predicted mel with a simple inverse transform.
model.eval();example="Welcome to Reading Assistant."
ids=torch.tensor([[vocab.get(p,1) for p in g2p(example.lower())][:MAX_PHONES]],dtype=torch.long)
with torch.no_grad():pred=model(ids)[0].T.clamp(min=0)
mel_power=torch.expm1(pred).clamp(min=0)
inverse=torchaudio.transforms.InverseMelScale(n_stft=513,n_mels=N_MELS,sample_rate=SAMPLE_RATE)
griffin=torchaudio.transforms.GriffinLim(n_fft=1024,hop_length=256,n_iter=32)
try:
    wav=griffin(inverse(mel_power)).unsqueeze(0)
    torchaudio.save(str(out/"sample.wav"),wav,SAMPLE_RATE)
    print("Sample saved:",out/"sample.wav")
except Exception as exc:
    print("Audio reconstruction failed; checkpoint still saved:",str(exc))
print("Artifacts:",[p.name for p in out.iterdir()])
# Optional: uncomment to save to Drive:
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil
# shutil.copytree(str(out),"/content/drive/MyDrive/ReadingAssistantModel",dirs_exist_ok=True)